# Lecture 6 — Choropleth Maps

Run the notebook top to bottom with Python, pandas and Plotly. The data loader reads the two supplied CSVs from the working directory or its `data` folder and uses embedded copies if the files are absent. The custom boundaries for Task 2 are fetched from GitHub.

In [ ]:
from io import StringIO
from pathlib import Path
from urllib.request import urlopen
import json

import pandas as pd
import plotly.express as px

# Embedded copies of the supplied CSVs make the notebook portable.
gapminder_csv = """Country,Continent,GDP_per_capita,Life_expectancy,Population,CO2_per_capita
Afghanistan,Asia,550,63,38000000,0.3
Australia,Oceania,54000,83,26000000,15.1
Bangladesh,Asia,2200,73,170000000,0.5
Brazil,Americas,8900,76,215000000,2.1
Canada,Americas,46000,82,38000000,14.2
China,Asia,12500,77,1400000000,7.4
Colombia,Americas,6400,77,51000000,1.7
DR Congo,Africa,550,61,95000000,0.1
Egypt,Africa,3500,72,103000000,2.1
Ethiopia,Africa,950,68,120000000,0.1
France,Europe,42000,83,68000000,4.5
Germany,Europe,50000,81,84000000,7.7
Ghana,Africa,2200,65,32000000,0.5
India,Asia,2200,70,1400000000,1.9
Indonesia,Asia,4100,72,275000000,2.1
Iran,Asia,5600,77,86000000,7.6
Italy,Europe,34000,83,60000000,5.3
Japan,Asia,40000,84,125000000,8.5
Kenya,Africa,1900,67,54000000,0.4
Malaysia,Asia,11000,76,33000000,8.1
Mexico,Americas,9900,75,130000000,3.8
Morocco,Africa,3200,77,37000000,1.6
Mozambique,Africa,480,60,32000000,0.1
Myanmar,Asia,1500,67,54000000,0.6
Netherlands,Europe,52000,82,18000000,8.1
Nigeria,Africa,2000,55,213000000,0.7
Norway,Europe,82000,83,5000000,7.5
Pakistan,Asia,1500,68,225000000,0.9
Peru,Americas,6200,76,33000000,1.6
Philippines,Asia,3500,72,111000000,1.1
Poland,Europe,17000,78,38000000,7.0
Russia,Europe,11000,73,144000000,11.4
Saudi Arabia,Asia,23000,75,35000000,18.7
South Africa,Africa,6100,64,59000000,6.4
South Korea,Asia,31000,83,52000000,11.4
Spain,Europe,29000,84,47000000,5.2
Sudan,Africa,800,66,44000000,0.4
Sweden,Europe,53000,83,10000000,3.5
Tanzania,Africa,1050,66,63000000,0.2
Thailand,Asia,6500,77,70000000,3.8
Turkey,Europe,9600,78,85000000,5.4
Uganda,Africa,780,64,47000000,0.1
UK,Europe,46000,81,68000000,5.1
USA,Americas,62000,79,335000000,14.2
Venezuela,Americas,7000,72,29000000,4.0
Vietnam,Asia,3500,76,97000000,3.2
Zimbabwe,Africa,1100,62,16000000,0.6"""
development_csv = """Country,Region,HDI,Life_Expectancy,GNI_per_capita,Fertility_Rate,Population_M,Urban_pct
Norway,Europe,0.957,82.4,67614,0.1,5.4,83
Switzerland,Europe,0.955,83.8,72051,0.1,8.6,74
Germany,Europe,0.942,80.9,56053,0.3,84.0,77
United Kingdom,Europe,0.929,81.4,46827,0.4,67.7,84
United States,North America,0.921,78.9,65118,0.5,332.0,83
France,Europe,0.903,82.7,44538,0.3,67.5,81
Japan,East Asia,0.919,84.3,40113,0.3,125.0,92
South Korea,East Asia,0.925,83.5,42328,0.3,51.7,82
China,East Asia,0.768,77.1,11819,0.5,1412.0,62
Brazil,Latin America,0.754,75.9,8921,1.2,215.0,88
Mexico,Latin America,0.758,75.0,10046,0.8,130.0,81
South Africa,Africa,0.713,64.1,6374,1.8,60.0,68
Egypt,Africa,0.731,73.0,3019,1.1,104.0,43
India,South Asia,0.633,70.2,2257,1.4,1407.0,36
Bangladesh,South Asia,0.661,72.9,1698,0.9,166.0,40
Nigeria,Africa,0.535,55.2,2097,2.5,220.0,53
Ethiopia,Africa,0.498,67.8,855,4.0,120.0,22
Pakistan,South Asia,0.544,67.3,1388,3.4,231.0,37
Afghanistan,South Asia,0.478,62.8,508,4.5,40.0,26
DR Congo,Africa,0.459,60.7,557,5.6,100.0,46"""

def load_csv(filename, embedded_csv):
    for directory in (Path.cwd(), Path.cwd() / "data", Path.cwd().parent / "data"):
        path = directory / filename
        if path.is_file():
            return pd.read_csv(path)
    return pd.read_csv(StringIO(embedded_csv))

## Task 1 — Life expectancy relative to the sample average

I used `px.choropleth` because this is a world map with country-level data and a built-in country geometry layer. A diverging scale uses orange for below-average values and purple for above-average values, with zero at the centre and equal limits on either side.

The supplied `gapminder.csv` contains 47 countries and has no year column. The reference is the unweighted mean of those 47 country values. Grey land indicates countries outside the sample.

In [ ]:
gapminder = load_csv("gapminder.csv", gapminder_csv)
# Match the supplied country names to ISO-3 codes.
country_codes = dict(
    zip(
        "Afghanistan|Australia|Bangladesh|Brazil|Canada|China|Colombia|DR Congo|Egypt|Ethiopia|France|Germany|Ghana|India|Indonesia|Iran|Italy|Japan|Kenya|Malaysia|Mexico|Morocco|Mozambique|Myanmar|Netherlands|Nigeria|Norway|Pakistan|Peru|Philippines|Poland|Russia|Saudi Arabia|South Africa|South Korea|Spain|Sudan|Sweden|Tanzania|Thailand|Turkey|Uganda|UK|USA|Venezuela|Vietnam|Zimbabwe".split("|"),
        "AFG AUS BGD BRA CAN CHN COL COD EGY ETH FRA DEU GHA IND IDN IRN ITA JPN KEN MYS MEX MAR MOZ MMR NLD NGA NOR PAK PER PHL POL RUS SAU ZAF KOR ESP SDN SWE TZA THA TUR UGA GBR USA VEN VNM ZWE".split(),
        strict=True,
    )
)
gapminder["ISO3"] = gapminder["Country"].map(country_codes)
assert gapminder["ISO3"].notna().all() and gapminder["ISO3"].is_unique

sample_mean = gapminder["Life_expectancy"].mean()
gapminder["Deviation"] = gapminder["Life_expectancy"] - sample_mean
regional_means = gapminder.groupby("Continent")["Deviation"].mean().sort_values()
lowest_region = regional_means.index[0]
limit = gapminder["Deviation"].abs().max()

fig_world = px.choropleth(
    gapminder,
    locations="ISO3",
    locationmode="ISO-3",
    color="Deviation",
    hover_name="Country",
    hover_data={"ISO3": False, "Continent": True, "Life_expectancy": ":.1f", "Deviation": ":+.1f"},
    color_continuous_scale="PuOr",
    range_color=(-limit, limit),
    color_continuous_midpoint=0,
    projection="natural earth",
    labels={"Life_expectancy": "Life expectancy (years)", "Deviation": "Years from average"},
    title=f"{lowest_region} falls furthest below the sample's life expectancy average",
)
fig_world.update_geos(showland=True, landcolor="#e6e6e6", showframe=False)
print(f"Sample mean ({len(gapminder)} countries): {sample_mean:.2f} years")
print(regional_means.round(2).to_string())
fig_world.show()

## Task 2 — Human Development Index in the European sample

Map the European countries in `world_development.csv` using custom country polygons.

### Task 2 — Design decisions

**Boundary source** · [johan/world.geo.json](https://github.com/johan/world.geo.json), using the individual country GeoJSON files.

**Chart type** · `px.choropleth` draws the custom country polygons and can zoom to their extent. The join uses the ISO-3 identifier in each GeoJSON feature's top-level `id` field.

**Colour scale** · Sequential Blues shows low to high HDI. The legend spans the observed European sample to make differences within that sample visible. HDI itself is on a 0–1 scale.

In [ ]:
development = load_csv("world_development.csv", development_csv)
europe = development.loc[development["Region"].eq("Europe")].copy()
europe["ISO3"] = europe["Country"].map({
    "Norway": "NOR",
    "Switzerland": "CHE",
    "Germany": "DEU",
    "United Kingdom": "GBR",
    "France": "FRA",
})
assert europe["ISO3"].notna().all() and europe["ISO3"].is_unique

boundary_root = "https://raw.githubusercontent.com/johan/world.geo.json/master/countries"
features = []
for code in europe["ISO3"]:
    with urlopen(f"{boundary_root}/{code}.geo.json", timeout=20) as response:
        features.extend(json.load(response)["features"])

europe_geojson = {"type": "FeatureCollection", "features": features}
first_feature = europe_geojson["features"][0]
print("Example GeoJSON properties:", first_feature["properties"])
print("Example top-level feature id:", first_feature["id"])
feature_ids = [feature["id"] for feature in features]
assert sorted(feature_ids) == sorted(europe["ISO3"]), "Country and boundary IDs must match one-to-one."

highest = europe.loc[europe["HDI"].idxmax()]
lowest = europe.loc[europe["HDI"].idxmin()]
fig_europe = px.choropleth(
    europe,
    geojson=europe_geojson,
    locations="ISO3",
    featureidkey="id",
    color="HDI",
    hover_name="Country",
    hover_data={"ISO3": False, "HDI": ":.3f", "Life_Expectancy": ":.1f"},
    color_continuous_scale="Blues",
    projection="mercator",
    labels={"Life_Expectancy": "Life expectancy (years)"},
    title=f"{highest['Country']} leads the five European countries in the HDI sample",
)
fig_europe.update_geos(fitbounds="locations", visible=False)
print(f"Highest: {highest['Country']} ({highest['HDI']:.3f})")
print(f"Lowest: {lowest['Country']} ({lowest['HDI']:.3f})")
fig_europe.show()